# RAG + Few-Shot 피드백 생성기 (텍스트 입력)

## 데이터 역할
| 데이터 | 폴더 | 역할 |
|---|---|---|
| 답변-피드백 쌍 | `data/feedback_data/` | **Few-Shot** — 이형 어투/스타일 학습 |
| 추가 데이터셋 | `data/rag_data/` | **RAG DB** — 이형 강의 지식/논리 근거 |

## 튜닝 히스토리 (발표자료 기준)
1. Few-Shot만 → 어투 맞음, 내용 부족
2. RAG 추가 → 논리 강화
3. 프롬프트 엔지니어링 → 이형 스타일 규칙 적용
4. **타임스탬프 + 비언어 규칙 추가 → 최종** (이 노트북 기준)

## 실행 순서
`01_rag_build.ipynb` 먼저 실행 → `chroma_db/` 생성 필요

In [ ]:
%pip install google-genai chromadb==0.6.3 sentence-transformers python-dotenv --quiet

---
## STEP 1 — 환경 설정

In [ ]:
import os, re, time, random
from pathlib import Path
from dotenv import load_dotenv

BASE       = Path(r'C:\Users\82105\OneDrive\바탕 화면\interview-coach')
PROJ       = BASE.parent / '프로젝트3(면접)'
FB_DIR     = BASE / 'data' / 'feedback_data'   # Few-Shot용
CHROMA_DIR = PROJ / 'chroma_db'                # RAG DB

load_dotenv(PROJ / '.env')
GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY', '')
if not GOOGLE_API_KEY:
    raise ValueError('.env 파일에 GOOGLE_API_KEY가 없습니다.')

MODEL = 'gemini-1.5-flash'   # 긴 컨텍스트(1M 토큰) 필요
print(f'설정 완료 | 모델: {MODEL}')

---
## STEP 2 — RAG 로드 (rag_data 기반 ChromaDB)

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='paraphrase-multilingual-MiniLM-L12-v2'
)
collection = chroma_client.get_collection('interview_rag', embedding_function=ef)
print(f'RAG 로드 완료: {collection.count()}개 청크')

def retrieve(answer: str, n: int = 4):
    results = collection.query(query_texts=[answer[:200]], n_results=n)
    docs    = results['documents'][0]
    sources = [m.get('source', '') for m in results['metadatas'][0]]
    return docs, sources

---
## STEP 3 — Few-Shot 예시 로드 (feedback_data)

In [ ]:
pairs = []  # {'interview': str, 'feedback': str}

# ── 1~5 ─────────────────────────────────────────────────────
for iv_f in sorted((FB_DIR / '1~5' / 'Interview_text').glob('*.txt')):
    fb_f = FB_DIR / '1~5' / 'Feedback_text' / iv_f.name
    if fb_f.exists():
        iv = iv_f.read_text(encoding='utf-8').strip()
        fb = fb_f.read_text(encoding='utf-8').strip()
        if iv and fb:
            pairs.append({'interview': iv, 'feedback': fb})

# ── 11~16 ────────────────────────────────────────────────────
for iv_f in sorted((FB_DIR / '11~16' / 'interview_text').glob('*.txt')):
    fb_f = FB_DIR / '11~16' / 'feedback_text' / iv_f.name
    if fb_f.exists():
        iv = iv_f.read_text(encoding='utf-8').strip()
        fb = fb_f.read_text(encoding='utf-8').strip()
        if iv and fb:
            pairs.append({'interview': iv, 'feedback': fb})

# ── 6~10 (인터뷰/피드백 혼합 파일) ───────────────────────────
text_dir = FB_DIR / '6~10' / 'text'
for iv_f in sorted(text_dir.glob('*인터뷰*.txt')):
    num = re.match(r'(\d+)', iv_f.name)
    if not num:
        continue
    n = num.group(1)
    iv = iv_f.read_text(encoding='utf-8').strip()
    fb_files = sorted(text_dir.glob(f'{n}_[0-9]*.txt'))
    fb = '\n\n'.join(f.read_text(encoding='utf-8').strip() for f in fb_files)
    if iv and fb:
        pairs.append({'interview': iv, 'feedback': fb})

print(f'Few-Shot 예시 로드: {len(pairs)}개 답변-피드백 쌍')

---
## STEP 4 — Gemini 클라이언트 + 프롬프트 (발표자료 4차 튜닝 최종본)

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GOOGLE_API_KEY)

# ── 발표자료 slide19 최종 프롬프트 ───────────────────────────
SYSTEM_PROMPT = """대기업 인사 전문가이자 커리어 코칭 전문가인 '인터뷰 킹 이형' 스타일의 면접 피드백 코치입니다.
말투는 친절할 수 있지만 평가는 차갑고 직설적이어야 합니다.
학습한 내용을 바탕으로 지원자의 답변에 대한 피드백을 출력합니다.

중요한 규칙:
- 후보자의 답변에만 피드백을 제공합니다.
- 후속 질문, 예상 질문 또는 추가 질문 목록을 생성하지 마십시오.
- 일반적이거나 진부하거나 '상식적인' 조언은 피하세요.
- 후보자의 답변에 제공된 실제 내용을 바탕으로 평가합니다.
- 약점을 지적할 때, 왜 그것들이 부족한지 그리고 실제 면접관이 그 특정 점을 어떻게 인식하는지 설명하세요.
- 강점은 진정으로 얻은 것일 때만 인정하고, 지나치게 칭찬하지 마세요.
- 학습된 내용을 기반으로 하되, 학습된 내용은 참고하는 형식으로만 하고 너무 똑같이 피드백하지 마세요.
- 학습된 내용에서 중요하게 판단하는 내용들을 종합해서 피드백하도록 합니다.

출력 형식:
- 섹션 제목이나 제목 없이 자연스럽고 대화적인 흐름으로 작성하세요.
- 첫 번째 문장부터 바로 평가를 시작하세요.
- 후보자의 답변 중 개선해야 할 부분에 대해 구체적이고 실행 가능한 방향으로 결론을 내립니다.
- 모든 답변은 한국어로 작성해야 합니다."""


def get_feedback(answer: str, n_shots: int = 5,
                 n_rag: int = 4, verbose: bool = True) -> dict:
    """
    RAG + Few-Shot 통합 피드백 (텍스트 입력)
    - Few-Shot: Q./A. 형식으로 이형 어투 학습
    - RAG: 이형 강의 지식으로 근거 강화
    """
    # 1. RAG 검색
    rag_docs, rag_sources = retrieve(answer, n=n_rag)
    rag_context = '\n\n'.join(
        f'[참고{i+1}] {doc}' for i, doc in enumerate(rag_docs)
    )
    if verbose:
        print(f'RAG 검색: {len(rag_docs)}개 청크 | 출처: {rag_sources}')

    # 2. Few-Shot contents 구성 — Q./A. 형식 (발표자료 slide12 방식)
    shots = random.sample(pairs, min(n_shots, len(pairs)))
    contents = []
    for shot in shots:
        contents.append(types.Content(
            role='user',
            parts=[types.Part.from_text(text=f'Q.\n{shot["interview"]}')]
        ))
        contents.append(types.Content(
            role='model',
            parts=[types.Part.from_text(text=f'A.\n{shot["feedback"]}')]
        ))

    # 3. 실제 질문 (RAG 참고자료 포함)
    contents.append(types.Content(
        role='user',
        parts=[types.Part.from_text(
            text=f'Q.\n{answer}\n\n'
                 f'--- 참고 자료 (면접왕 이형 강의) ---\n{rag_context}'
        )]
    ))

    if verbose:
        print(f'Few-Shot {n_shots}개 + RAG {n_rag}개 → Gemini 호출 중...')

    # 4. Gemini 호출
    resp = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0.4,
            max_output_tokens=1200
        )
    )

    if verbose:
        print('\n' + '━' * 60)
        print(resp.text)
        print('━' * 60)

    return {
        'answer': answer, 'feedback': resp.text,
        'rag_sources': rag_sources, 'n_shots': n_shots
    }

print('준비 완료')

---
## STEP 5 — 피드백 받기
> **여기서 답변을 수정하고 실행하세요.**

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ANSWER = """안녕하세요. 저는 성실하고 책임감 있는 사람입니다.
학교에서 열심히 공부했고, 팀 프로젝트도 많이 해봤습니다.
이 회사에 오고 싶어서 지원했습니다. 잘 부탁드립니다."""

N_SHOTS = 5   # Few-Shot 예시 수 (어투 학습, 5~15 권장)
N_RAG   = 4   # RAG 청크 수 (근거 강화, 3~5 권장)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

result = get_feedback(ANSWER, n_shots=N_SHOTS, n_rag=N_RAG)